In [1]:
from dotenv import load_dotenv
load_dotenv() 

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """read an email from the given address."""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """send an email to the given address with the given subject and body."""
    return f"Email sent."

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ]
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Read my email and send a response to the sender. You can think of the response on your own.")],
        "email": "Hellos, I am unable to make the meeting tomorrow. Can we reschedule? Regards, Ash."
    },
    config = config
)

In [5]:
from pprint import pprint
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'Ash,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all! '
                                                                          'I '
                                                                          'understand '
                                                                          'that '
                                                                          'things '
                                                                          'come '
                

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi Ash,\n\nNo problem at all! I understand that things come up. I'd be happy to reschedule our meeting.\n\nCould you let me know what days and times work best for you in the coming week? I'm flexible and can adjust my schedule to accommodate your availability.\n\nLooking forward to connecting soon.\n\nBest regards"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi Ash,\\n\\nNo problem at all! I understand that things come up. I\'d be happy to reschedule our meeting.\\n\\nCould you let me know what days and times work best for you in the coming week? I\'m flexible and can adjust my schedule to accommodate your availability.\\n\\nLooking forward to connecting soon.\\n\\nBest regards"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='68c143e9b402097410dfbdce49ff533c')]


In [7]:
# access the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Ash,

No problem at all! I understand that things come up. I'd be happy to reschedule our meeting.

Could you let me know what days and times work best for you in the coming week? I'm flexible and can adjust my schedule to accommodate your availability.

Looking forward to connecting soon.

Best regards


In [8]:
# in case rejection is required. 
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume = {
            "decisions": [{
                "type": "reject",
                "message": "No please sign off - Let off the hook...for now, Zain"
            }]
        }
    ),
    config = config
)

print(response)

{'messages': [HumanMessage(content='Read my email and send a response to the sender. You can think of the response on your own.', additional_kwargs={}, response_metadata={}, id='39cfd671-c566-44ed-9db7-6d50113ec485'), AIMessage(content=[{'text': "I'll read your email first and then send a thoughtful response.", 'type': 'text'}, {'id': 'toolu_01HbjAvBuGSEBL7mRYPFj8db', 'input': {}, 'name': 'read_email', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_016cmWmvW47EEDSwRyxiqVvx', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 626, 'output_tokens': 51, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--